# Turning Signal Into Concentration: Calibration, Detection Limits, and Honest Uncertainty
### 4.2 — UV-Vis: Building Your First Calibration Curve

An instrument never measures concentration. It measures a **signal** — an
absorbance, a peak height, an area — and your job is to run that backwards:
*given the signal, how much analyte was in the sample?* A **calibration curve**
is the tool that makes that inversion possible. You measure a handful of
standards whose concentrations you already know, find the relationship between
signal and concentration, and then use it to read concentration off the signal
of an unknown.

It is tempting to think this lesson is "how to fit a straight line." It isn't.
`numpy.polyfit(x, y, 1)` is one line of code and it will *always* return a slope
and an intercept — even when a straight line is the wrong model and the number
you read back is quietly wrong. The real content of calibration is the reasoning
*around* the line:

- **what a calibration curve means scientifically** — why a signal should scale
  with concentration at all (Beer–Lambert), and what the slope and intercept are
  telling you;
- **the fit** — slope, intercept, and R², and what each one does and does not
  promise;
- **residuals** — why a high R² alone is *not* evidence of a good calibration;
- **predicting an unknown** — and, just as important, the **uncertainty** on
  that predicted concentration;
- **the blank** — why the noise of a zero-analyte sample sets the floor of what
  you can detect, and how that becomes **LOD** and **LOQ**;
- **when calibration fails** — how a curved response can hide behind a great-
  looking R², and how a bad choice of range produces confident, wrong answers.

Because every spectrum here comes from `uvvis.simulate()`, we always **know the
true concentration** behind each measurement. So we can *grade* every fit,
prediction, and detection limit against the right answer — something a real
calibration can never offer.

> **Where this sits:** this builds on **4.1 (Beer–Lambert and the Absorbance
> Mindset)** and on the signal-processing tools from Track 3 — **3.3 (baseline
> correction)**, **3.4 (peak detection)**, and especially **3.5 (peak
> integration)**, where we first turned a band into a number and fit a quick
> calibration. Here that single idea becomes a careful, reportable method.

**What we'll cover**

1. Setup
2. What a calibration curve actually means
3. Building a realistic calibration series (standards, replicates, blanks)
4. From a spectrum to one number — height and area
5. Fitting the line — slope, intercept, R²
6. Residuals — why R² alone is not enough
7. Predicting an unknown — and its uncertainty
8. The blank, the detection limit (LOD), and the quantitation limit (LOQ)
9. When calibration fails — nonlinearity and misleading confidence
10. A reusable `calibrate()` / `predict_concentration()` helper

## 1. Setup

In [ ]:
# Standard scientific Python stack.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# np.trapezoid is the trapezoidal rule (our integrator from 3.5). Older NumPy
# spells it np.trapz; alias it so the notebook runs either way.
try:
    trapezoid = np.trapezoid          # NumPy >= 2.0
except AttributeError:
    trapezoid = np.trapz              # older NumPy

# We use Student's t for honest confidence intervals (small numbers of points).
from scipy import stats

# Our UV-Vis simulator. It is a faithful Beer-Lambert model: band height scales
# with concentration, the baseline does not, and each dataset carries its own
# ground-truth concentrations in `meta`. That ground truth is our answer key.
from simulated_data import uvvis

# A folder for saved figures (regenerable scratch; git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

# One consistent, readable plot style for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# A small, consistent colour vocabulary so every figure reads the same way.
C_SIG   = "#1a73e8"  # the measured signal / standards (blue)
C_TRUE  = "#188038"  # ground truth / the fitted model (green)
C_FALSE = "#ea4335"  # error / bias / a problem (red)
C_AUX   = "#f9ab00"  # helper lines, limits, baselines (amber)
C_FILL  = "#aecbfa"  # shaded regions (light blue)

print("Ready.")

### Our analyte and how we measure it

Throughout the notebook a single chromophore stands in for "the thing we want to
quantify": one clean absorbance band at **520 nm** with a **40 nm** FWHM. At unit
concentration its height is **1.0 AU**; Beer–Lambert means the height scales
linearly with concentration, so concentration `0.5` gives a `0.5 AU` band.

To turn a spectrum into one number we reuse the recipe from **3.5**: take a
window around the band (`±1.5 · FWHM`, wide enough to reach the baseline on both
sides), subtract a **local linear baseline** drawn between the window endpoints,
then read off either the **height** (the corrected peak maximum) or the **area**
(the trapezoidal integral of the corrected band). Doing the baseline subtraction
*inside* the measurement is what keeps a small instrumental offset from leaking
into our calibration.

In [ ]:
# --- The analyte band: (center_nm, FWHM_nm, amplitude_at_unit_concentration) ---
ANALYTE = (520.0, 40.0, 1.0)
CENTER, FWHM = ANALYTE[0], ANALYTE[1]

# A realistic, fixed measurement context so every standard is treated identically.
NOISE = 0.004                                   # detector noise (sigma, in AU)
BASELINE = {"type": "sloping", "slope": 0.0, "offset": 0.02}  # small flat offset

# Integration/measurement window: wide enough that the endpoints sit on baseline.
LO, HI = CENTER - 1.5 * FWHM, CENTER + 1.5 * FWHM


def _corrected_window(x, y):
    '''Return the window x-values and the band after a local linear baseline drop.

    The baseline is the straight line connecting the signal at the two window
    endpoints; subtracting it removes a flat or gently sloping background so the
    measurement reflects the band, not the offset under it.
    '''
    win = (x >= LO) & (x <= HI)
    xs, ys = x[win], y[win]
    base = np.interp(xs, [xs[0], xs[-1]], [ys[0], ys[-1]])
    return xs, ys - base


def peak_height(x, y):
    '''Baseline-corrected peak height (AU) -- one number for one spectrum.'''
    _, corrected = _corrected_window(x, y)
    return corrected.max()


def peak_area(x, y):
    '''Baseline-corrected peak area (AU*nm) by the trapezoidal rule.'''
    xs, corrected = _corrected_window(x, y)
    return trapezoid(corrected, xs)


def measure(concentration, seed, noise=NOISE, stray=0.0):
    '''Simulate one standard at a known concentration and measure it.

    Returns (height, area). `stray` injects a small stray-light nonlinearity
    used only in section 9; it is 0 (perfectly linear) everywhere else.
    '''
    ds = uvvis.simulate(peaks=[ANALYTE], concentration=concentration,
                        noise=noise, baseline=BASELINE, seed=seed)
    x, y = ds.x, ds.single()
    if stray > 0:
        # Stray light flattens absorbance at high values (see section 9).
        y = -np.log10((1.0 - stray) * 10.0**(-y) + stray)
    return peak_height(x, y), peak_area(x, y)


# Quick check: a unit-concentration standard should give a ~1.0 AU height.
h1, a1 = measure(1.0, seed=42)
print(f"Unit-concentration standard: height = {h1:.3f} AU, area = {a1:.1f} AU*nm")

## 2. What a calibration curve actually means

Before fitting anything, it is worth being clear about *why* a straight line
should appear at all — because that reason is also where calibration eventually
breaks.

For UV-Vis, the physics is the **Beer–Lambert law**:

$$A = \varepsilon\, b\, c$$

Absorbance $A$ is proportional to the analyte concentration $c$, with the
proportionality set by the molar absorptivity $\varepsilon$ (a property of the
molecule) and the path length $b$ (a property of the cuvette). Hold the molecule
and the cuvette fixed and absorbance is simply **a constant times concentration**
— a straight line through the origin.

So why calibrate at all, if physics already promises a line? Because we almost
never know $\varepsilon b$ from first principles, the instrument adds its own
small offset and noise, and the law only holds over a limited range. Calibration
*measures* the line for our analyte on our instrument today:

- the **slope** is the **sensitivity** — how much signal we get per unit
  concentration (it is $\varepsilon b$ for the height of an absorbance band);
- the **intercept** is the signal we read at zero concentration — ideally near
  zero once the blank is handled, and a flag for a background problem if not.

A calibration curve, then, is an *empirical, local* restatement of Beer–Lambert
that we can actually invert: measure standards, fit signal-vs-concentration,
then run it backwards to read concentration off an unknown's signal.

## 3. Building a realistic calibration series

A calibration is only as good as the standards behind it. A realistic series has
three things our simulator lets us reproduce exactly:

- **known concentrations** spanning the range we care about — here six levels
  from `0.1` to `1.0` (in arbitrary concentration units);
- **replicates** at each level — three independent measurements, because a
  single reading hides how much the signal naturally scatters;
- **blanks** — samples with *no analyte* (concentration `0`), which we will need
  in section 8 to find the detection limit.

Each standard gets its own random seed, so the replicates differ only by genuine
measurement noise — exactly as if we had prepared and read the same solution
three times.

In [ ]:
# Six concentration levels, three replicates each. Each measurement gets a unique
# seed so replicates differ only by realistic detector noise.
levels = np.array([0.1, 0.2, 0.4, 0.6, 0.8, 1.0])
N_REPLICATES = 3

records = []
seed = 0
for level in levels:
    for rep in range(N_REPLICATES):
        h, a = measure(level, seed=seed)
        records.append({"concentration": level, "replicate": rep + 1,
                        "height": h, "area": a})
        seed += 1

standards = pd.DataFrame(records)
print(standards.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

Look at the `height` column within any one concentration: the three replicates
agree to a few thousandths of an AU but are not identical — that spread *is* the
measurement noise, and it is what will ultimately limit both our detection limit
and the uncertainty on an unknown. Now let's see the spectra those numbers came
from.

In [ ]:
# Show one spectrum per concentration level so the calibration is visible as
# physics first, numbers second: taller band = more analyte.
fig, ax = plt.subplots()
for level in levels:
    ds = uvvis.simulate(peaks=[ANALYTE], concentration=level,
                        noise=NOISE, baseline=BASELINE, seed=int(level * 100))
    ax.plot(ds.x, ds.single(), label=f"c = {level:.1f}")

ax.axvspan(LO, HI, color=C_FILL, alpha=0.25, label="measurement window")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Absorbance (AU)")
ax.set_title("Calibration standards: band height grows with concentration")
ax.legend(ncol=2, fontsize=9)
fig.tight_layout(); fig.savefig(EXPORTS / "01_cal_series_spectra.png", dpi=130)
plt.show()

## 4. From a spectrum to one number — height and area

A calibration curve plots **one number per standard** against concentration, so
first we have to collapse each spectrum to a single value. Two honest choices
(both from lesson 3.5):

- **peak height** — the corrected band maximum. Simple, fast, and proportional
  to concentration as long as the band *shape* never changes.
- **peak area** — the integral of the corrected band. More robust when the band
  broadens or shifts slightly run-to-run, because area conserves "how much"
  even as the shape moves.

Either works for a stable UV-Vis band. We compute both and confirm each one
rises linearly with concentration; the rest of the notebook uses **height** for
concreteness.

In [ ]:
# Plot height and area against concentration. Both should be straight lines; the
# point is that "which number" is a choice, and both are valid here.
fig, (axh, axa) = plt.subplots(1, 2, figsize=(11, 4.2))

axh.scatter(standards["concentration"], standards["height"],
            color=C_SIG, zorder=3)
axh.set_xlabel("concentration (arb. units)"); axh.set_ylabel("peak height (AU)")
axh.set_title("Height vs concentration")

axa.scatter(standards["concentration"], standards["area"],
            color=C_SIG, zorder=3)
axa.set_xlabel("concentration (arb. units)"); axa.set_ylabel("peak area (AU*nm)")
axa.set_title("Area vs concentration")

fig.tight_layout(); fig.savefig(EXPORTS / "02_height_vs_area.png", dpi=130)
plt.show()

Both panels are straight lines through (almost) the origin — exactly what
Beer–Lambert predicts. The replicate points at each concentration sit nearly on
top of one another, so the scatter is small but real. From here we work with
**height**; everything that follows applies identically to area.

## 5. Fitting the line — slope, intercept, R²

Now we fit signal-vs-concentration with **ordinary least squares** — the line
that minimises the sum of squared vertical distances from the points. We fit on
**all** the replicate points (not the per-level means): using every measurement
lets the fit "see" the true scatter, which we will need for honest uncertainty
later.

Three numbers come out, and it is worth naming what each one is for:

- **slope** $m$ — sensitivity (signal per unit concentration). For our band the
  true value is `1.0 AU` per unit, because a unit-concentration band is `1.0 AU`
  tall.
- **intercept** $b$ — the modelled signal at zero concentration. We *want* this
  near zero; a large intercept means an uncorrected background.
- **R²** — the fraction of the signal's variation the line explains. Close to 1
  is necessary but, as section 6 shows, **not sufficient**.

In [ ]:
def fit_line(x, y):
    '''Least-squares line y = slope*x + intercept, with the statistics a
    calibration actually needs.

    Returns a dict with slope, intercept, R^2, the residual standard deviation
    s_y/x (the typical vertical miss of a point from the line), Sxx (the spread
    of the x-values), and the standard errors of slope and intercept.
    '''
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    slope, intercept = np.polyfit(x, y, 1)
    y_hat = slope * x + intercept

    ss_res = np.sum((y - y_hat) ** 2)              # unexplained variation
    ss_tot = np.sum((y - y.mean()) ** 2)           # total variation
    r2 = 1.0 - ss_res / ss_tot

    s_yx = np.sqrt(ss_res / (n - 2))               # residual std dev (n-2 d.o.f.)
    Sxx = np.sum((x - x.mean()) ** 2)
    se_slope = s_yx / np.sqrt(Sxx)
    se_intercept = s_yx * np.sqrt(np.sum(x ** 2) / (n * Sxx))

    return {"slope": slope, "intercept": intercept, "r2": r2, "s_yx": s_yx,
            "Sxx": Sxx, "n": n, "x_mean": x.mean(), "y_mean": y.mean(),
            "se_slope": se_slope, "se_intercept": se_intercept}


cal = fit_line(standards["concentration"], standards["height"])

print(f"slope      = {cal['slope']:.4f} +/- {cal['se_slope']:.4f} AU per conc-unit")
print(f"intercept  = {cal['intercept']:.4f} +/- {cal['se_intercept']:.4f} AU")
print(f"R^2        = {cal['r2']:.5f}")
print(f"s_y/x      = {cal['s_yx']:.4f} AU   (typical vertical scatter about the line)")
print(f"true slope = 1.0000        (a unit-concentration band is 1.0 AU tall)")

In [ ]:
# The calibration line through the standards.
fig, ax = plt.subplots()
ax.scatter(standards["concentration"], standards["height"],
           color=C_SIG, zorder=3, label="standards (3 replicates each)")
xs = np.linspace(0, 1.05, 50)
ax.plot(xs, cal["slope"] * xs + cal["intercept"], color=C_TRUE,
        label=(f"fit: signal = {cal['slope']:.3f}c + {cal['intercept']:.3f}"
               f"   (R² = {cal['r2']:.4f})"))
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("peak height (AU)")
ax.set_title("Calibration curve (peak height)")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "03_calibration_fit.png", dpi=130)
plt.show()

The recovered slope (`~0.99`) sits right on the true sensitivity of `1.0`, the
intercept is within a few thousandths of zero (our small instrumental offset was
removed by the in-window baseline subtraction), and R² is essentially 1. This is
what a *good* calibration looks like — but R² being near 1 is the easiest thing
to fake, so we should never stop here.

## 6. Residuals — why R² alone is not enough

R² answers "how much of the signal's variation does the line explain?" It does
*not* answer "is a line the right model?" A gently curved response can post a
gorgeous R² and still be systematically wrong — we will see exactly that in
section 9.

The tool that catches it is the **residual plot**: the leftover gap between each
point and the line, $r_i = y_i - \hat{y}_i$, plotted against concentration. The
rule is simple:

- **good fit → residuals are structureless** — random scatter above and below
  zero, no trend, roughly constant spread;
- **wrong model → residuals show a pattern** — a curve, a fan, a drift.

For our good calibration the residuals should be pure noise. Let's confirm,
because seeing what "no structure" looks like is what makes the broken case in
section 9 obvious.

In [ ]:
# Residuals of the good calibration: each point minus the fitted line.
resid = standards["height"] - (cal["slope"] * standards["concentration"] + cal["intercept"])

fig, ax = plt.subplots()
ax.axhline(0, color=C_TRUE, lw=1)
ax.scatter(standards["concentration"], resid, color=C_SIG, zorder=3)
# +/- one residual standard deviation, the typical scatter band.
ax.axhspan(-cal["s_yx"], cal["s_yx"], color=C_FILL, alpha=0.3,
           label="±1 s_y/x")
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("residual (AU)")
ax.set_title("Residuals of the linear calibration — no pattern is the goal")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "04_residuals_good.png", dpi=130)
plt.show()

print(f"residuals scatter randomly within about ±{cal['s_yx']:.4f} AU, no trend.")

No smile, no frown, no fan — the points fall on both sides of zero with no
concentration-dependent pattern. *That* is what earns the high R². Keep this
picture in mind: in section 9 the R² will look just as good while this plot
quietly bends.

## 7. Predicting an unknown — and its uncertainty

Now the payoff: an **unknown** sample. We measure its signal $y_0$ and invert the
calibration line to get its concentration:

$$x_0 = \frac{y_0 - b}{m}$$

A point estimate alone is not a result, though — we need to know *how sure* we
are. The standard error of an inverse prediction (Miller & Miller, the classic
analytical-chemistry formula) is

$$s_{x_0} = \frac{s_{y/x}}{m}\sqrt{\frac{1}{M} + \frac{1}{N} +
\frac{(y_0-\bar{y})^2}{m^2\,S_{xx}}}$$

where $M$ is the number of replicate readings of the unknown, $N$ the number of
calibration points, and $S_{xx}$ the spread of the calibration concentrations.
The formula rewards three things you control: **more replicates of the unknown**
($M$), **more calibration standards** ($N$), and an unknown that falls **near the
centre of the calibrated range** (the last term grows as $y_0$ moves toward the
ends). We turn the standard error into a 95% interval with Student's $t$.

We will measure an unknown that is *secretly* at concentration `0.55`, so we can
check both the estimate and the interval against the truth.

In [ ]:
def predict_concentration(cal, signal_values, confidence=0.95):
    '''Invert a calibration line to a concentration, with a confidence interval.

    signal_values : one or more replicate signal readings of the same unknown.
    Returns (x0, half_width): the estimated concentration and the +/- half-width
    of the confidence interval at the requested level.
    '''
    y0 = np.mean(signal_values)
    M = len(np.atleast_1d(signal_values))
    x0 = (y0 - cal["intercept"]) / cal["slope"]

    # Standard error of the inverse prediction (Miller & Miller).
    se = (cal["s_yx"] / cal["slope"]) * np.sqrt(
        1.0 / M + 1.0 / cal["n"]
        + (y0 - cal["y_mean"]) ** 2 / (cal["slope"] ** 2 * cal["Sxx"])
    )
    t = stats.t.ppf(0.5 + confidence / 2.0, cal["n"] - 2)
    return x0, t * se


# Measure the unknown three times (its true concentration is 0.55).
TRUE_UNKNOWN = 0.55
unknown_signals = [measure(TRUE_UNKNOWN, seed=5000 + r)[0] for r in range(3)]

x0, half = predict_concentration(cal, unknown_signals)
print(f"unknown signals (AU): {[f'{v:.4f}' for v in unknown_signals]}")
print(f"predicted concentration = {x0:.3f} ± {half:.3f}  (95% CI)")
print(f"true concentration      = {TRUE_UNKNOWN}")
print(f"recovery                = {100 * x0 / TRUE_UNKNOWN:.1f}%")
print(f"true value inside CI?   {abs(x0 - TRUE_UNKNOWN) <= half}")

In [ ]:
# Show the prediction on the calibration: read the signal across to the line,
# then down to the concentration axis, with the confidence interval marked.
fig, ax = plt.subplots()
ax.scatter(standards["concentration"], standards["height"],
           color=C_SIG, zorder=3, label="standards")
xs = np.linspace(0, 1.05, 50)
ax.plot(xs, cal["slope"] * xs + cal["intercept"], color=C_TRUE, label="calibration")

y0 = np.mean(unknown_signals)
ax.axhline(y0, color=C_AUX, ls="--", lw=1)
ax.plot([x0, x0], [0, y0], color=C_FALSE, ls="--", lw=1)
ax.axvspan(x0 - half, x0 + half, color=C_FALSE, alpha=0.15, label="95% CI")
ax.scatter([x0], [y0], color=C_FALSE, zorder=4,
           label=f"unknown: {x0:.2f} ± {half:.2f}")
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("peak height (AU)")
ax.set_title("Reading an unknown off the calibration")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "05_prediction.png", dpi=130)
plt.show()

The estimate lands within a fraction of a percent of the true `0.55`, and the
true value sits comfortably inside the 95% interval — as it should. Notice the
interval is a real, reportable number, not a decoration: it shrinks if we run
more replicates of the unknown, and it would widen if the unknown sat near the
edge of (or beyond) the calibrated range. A concentration without an uncertainty
is only half a result.

## 8. The blank, the detection limit (LOD), and the quantitation limit (LOQ)

Every method has a floor. Below some concentration the analyte's signal is lost
in the noise of a sample that contains *none* of it — the **blank**. Two limits
formalise that floor:

- **LOD — limit of detection**: the smallest concentration we can confidently
  say is *present* (distinguishable from the blank), but not reliably quantify.
- **LOQ — limit of quantitation**: the smallest concentration we can report with
  acceptable precision.

The standard definitions tie both to the **blank noise** $\sigma_{blank}$ and the
calibration **slope** (which converts a signal level into a concentration):

$$\text{LOD} = \frac{3.3\,\sigma_{blank}}{m}, \qquad
\text{LOQ} = \frac{10\,\sigma_{blank}}{m}$$

The factor 3.3 makes the detection decision at roughly 99% confidence; 10 sets
quantitation at about a 10% relative standard deviation. We need $\sigma_{blank}$,
so we measure a stack of blanks and take their standard deviation.

In [ ]:
# Measure 20 independent blanks (concentration = 0) the same way we measure a
# standard. Their spread is the noise floor of the method.
blank_signals = np.array([measure(0.0, seed=1000 + r)[0] for r in range(20)])
sigma_blank = blank_signals.std(ddof=1)
mean_blank = blank_signals.mean()

slope = cal["slope"]
lod_blank = 3.3 * sigma_blank / slope
loq_blank = 10.0 * sigma_blank / slope

print(f"blank signal: mean = {mean_blank:.4f} AU, std (sigma_blank) = {sigma_blank:.4f} AU")
print(f"LOD = 3.3 * sigma_blank / slope = {lod_blank:.4f}  conc-units")
print(f"LOQ = 10  * sigma_blank / slope = {loq_blank:.4f}  conc-units")

### A second route: the calibration's own scatter

When clean blanks are awkward to measure, a common alternative uses the
**residual standard deviation** of the calibration, $s_{y/x}$, in place of
$\sigma_{blank}$ — it is the calibration's own estimate of the noise. The two
routes rarely agree to the last digit, and that is the lesson: a detection limit
is an **estimate that depends on its definition**, not a single hard number.

In [ ]:
lod_reg = 3.3 * cal["s_yx"] / slope
loq_reg = 10.0 * cal["s_yx"] / slope

summary = pd.DataFrame({
    "method": ["blank std (sigma_blank)", "calibration residual (s_y/x)"],
    "noise (AU)": [sigma_blank, cal["s_yx"]],
    "LOD (conc)": [lod_blank, lod_reg],
    "LOQ (conc)": [loq_blank, loq_reg],
})
print(summary.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"Our lowest standard was at concentration {levels.min()}, which sits "
      f"{'above' if levels.min() > loq_blank else 'below'} the LOQ — good practice "
      "is to keep every quantified standard at or above the LOQ.")

In [ ]:
# Visualise the floor: blanks, the LOD and LOQ as concentration thresholds, and
# the low end of the calibration.
fig, ax = plt.subplots()
xs = np.linspace(0, 0.30, 50)
ax.plot(xs, slope * xs + cal["intercept"], color=C_TRUE, label="calibration")
ax.axhspan(mean_blank - sigma_blank, mean_blank + sigma_blank,
           color=C_AUX, alpha=0.25, label="blank ±1σ")
ax.axvline(lod_blank, color=C_FALSE, ls="--", label=f"LOD = {lod_blank:.3f}")
ax.axvline(loq_blank, color=C_SIG, ls="--", label=f"LOQ = {loq_blank:.3f}")
ax.scatter(np.zeros_like(blank_signals), blank_signals,
           color="grey", alpha=0.6, zorder=3, label="blanks")
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("peak height (AU)")
ax.set_title("Detection (LOD) and quantitation (LOQ) limits sit just above the blank noise")
ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(EXPORTS / "06_blank_lod_loq.png", dpi=130)
plt.show()

Both limits live just above the blank scatter. A reading between the blank and
the LOD is "not detected"; between LOD and LOQ it is "detected but not reliably
quantifiable" (report it as `< LOQ`, not as a number you would defend); at or
above LOQ you can report a value with its uncertainty. Crucially, **LOD scales
with `1/slope`** — anything that raises sensitivity (longer path length, a
stronger chromophore, less noise) lowers the detection limit.

## 9. When calibration fails — nonlinearity and misleading confidence

Everything so far worked because Beer–Lambert held. Real calibrations bend,
especially at **high concentration**, where stray light, instrument saturation,
and chemical effects pull the response below the linear prediction. The danger is
that the bend can be subtle enough that **R² still looks excellent** while the
calibration is quietly biased.

Our simulator is a faithful *linear* Beer–Lambert model, so to teach failure we
inject a known, realistic deviation by hand: a small **stray-light** fraction
$s$, which makes the *observed* absorbance flatten at high values,

$$A_{obs} = -\log_{10}\!\big[(1-s)\,10^{-A} + s\big].$$

At low absorbance $A_{obs}\approx A$ (linear); at high absorbance it saturates
toward $-\log_{10}(s)$. We extend the range to a concentration of `2.5`, fit a
straight line through everything, and look — first at R², then at the residuals.

In [ ]:
# Standards over a wider range, now with 0.3% stray light injected.
STRAY = 0.003
levels_wide = np.linspace(0.1, 2.5, 8)
heights_wide = np.array([measure(c, seed=8000 + k, stray=STRAY)[0]
                         for k, c in enumerate(levels_wide)])

bad = fit_line(levels_wide, heights_wide)
print(f"linear fit through the curved data:")
print(f"  slope = {bad['slope']:.4f}   intercept = {bad['intercept']:.4f}")
print(f"  R^2   = {bad['r2']:.4f}   <- still looks great!")

In [ ]:
# The fit looks fine on top of the points...
fig, ax = plt.subplots()
ax.scatter(levels_wide, heights_wide, color=C_SIG, zorder=3, label="standards (curved response)")
xs = np.linspace(0, 2.6, 50)
ax.plot(xs, bad["slope"] * xs + bad["intercept"], color=C_FALSE,
        label=f"linear fit (R² = {bad['r2']:.4f})")
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("peak height (AU)")
ax.set_title("A linear fit through a curved response — R² says 'great'")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "07_nonlinear_fit.png", dpi=130)
plt.show()

In [ ]:
# ...until you look at the residuals, where the curvature is unmistakable.
resid_bad = heights_wide - (bad["slope"] * levels_wide + bad["intercept"])

fig, ax = plt.subplots()
ax.axhline(0, color=C_TRUE, lw=1)
ax.plot(levels_wide, resid_bad, "o-", color=C_FALSE, zorder=3)
ax.set_xlabel("concentration (arb. units)")
ax.set_ylabel("residual (AU)")
ax.set_title("The residual plot exposes what R² hid: a clear frown (curvature)")
fig.tight_layout(); fig.savefig(EXPORTS / "08_nonlinear_residuals.png", dpi=130)
plt.show()

print(f"R^2 = {bad['r2']:.4f}, yet the residuals march negative -> positive -> "
      "negative.")
print("That arch is systematic error, not noise. R^2 alone would never tell you.")

This is the whole point of section 6 paying off. The R² is `0.994` — the kind of
number you would happily put in a report — but the residuals trace a clean arch
(positive in the middle, negative at both ends). That structure is the signature
of fitting a straight line to a curve: the model is **systematically wrong**, and
no amount of R² will warn you. The fix is to **restrict the calibration to the
linear range** (or fit a curve / dilute the sample). Restricting works here:

In [ ]:
# Keep only the standards in the linear region (concentration <= 1.2) and refit.
lin_mask = levels_wide <= 1.2
good_range = fit_line(levels_wide[lin_mask], heights_wide[lin_mask])
print(f"restricted to c <= 1.2:  R^2 = {good_range['r2']:.5f}, "
      f"slope = {good_range['slope']:.4f}")
resid_lin = heights_wide[lin_mask] - (good_range['slope'] * levels_wide[lin_mask]
                                      + good_range['intercept'])
print(f"residuals now within ±{np.abs(resid_lin).max():.4f} AU and unstructured.")

# But beware extrapolating the restricted line past where it was validated:
for c_high in [2.0, 2.5]:
    y_high = measure(c_high, seed=9500 + int(c_high * 10), stray=STRAY)[0]
    c_pred = (y_high - good_range["intercept"]) / good_range["slope"]
    print(f"  extrapolate to c = {c_high}: predicts {c_pred:.2f} "
          f"(recovery {100 * c_pred / c_high:.0f}%) <- biased low, outside the range")

Two lessons land together. First, **a high R² is not a certificate of
linearity** — only a structureless residual plot is. Second, even a perfect
linear calibration is only valid **inside the range you measured**: extrapolating
the restricted line up to `2.5` recovers only ~89% of the true amount, because the
real response has bent away. "Confident and wrong" is exactly what a good-looking
fit over the wrong range delivers.

## 10. A reusable `calibrate()` / `predict_concentration()` helper

Let's bottle the workflow so later lessons (and real work) can lean on it. We
already wrote `fit_line` and `predict_concentration`; here we wrap them into a
single `calibrate()` that also computes LOD/LOQ, and demonstrate the full
round-trip on a fresh, blind unknown.

In [ ]:
def calibrate(concentrations, signals, blank_signals=None):
    '''Fit a calibration and bundle everything needed to use and report it.

    Returns the fit dict from fit_line, plus LOD and LOQ. If blank_signals are
    given, the limits use the blank standard deviation; otherwise they fall back
    to the calibration residual s_y/x.
    '''
    calf = dict(fit_line(concentrations, signals))
    noise = (np.std(blank_signals, ddof=1) if blank_signals is not None
             else calf["s_yx"])
    calf["noise_source"] = "blank std" if blank_signals is not None else "s_y/x"
    calf["lod"] = 3.3 * noise / calf["slope"]
    calf["loq"] = 10.0 * noise / calf["slope"]
    return calf


# Build the calibration from our standards + blanks, then read a blind unknown.
model = calibrate(standards["concentration"], standards["height"],
                  blank_signals=blank_signals)

TRUE_BLIND = 0.37                      # the answer key, hidden from the model
blind_signals = [measure(TRUE_BLIND, seed=6000 + r)[0] for r in range(3)]
x_pred, x_ci = predict_concentration(model, blind_signals)

print("--- calibration report ---")
print(f"slope      : {model['slope']:.4f} ± {model['se_slope']:.4f} AU/conc")
print(f"intercept  : {model['intercept']:.4f} ± {model['se_intercept']:.4f} AU")
print(f"R^2        : {model['r2']:.5f}")
print(f"LOD / LOQ  : {model['lod']:.3f} / {model['loq']:.3f} conc  "
      f"(from {model['noise_source']})")
print(f"range      : {levels.min()} – {levels.max()} conc-units")
print("--- unknown ---")
print(f"predicted  : {x_pred:.3f} ± {x_ci:.3f} (95% CI)")
print(f"true       : {TRUE_BLIND}    recovery = {100 * x_pred / TRUE_BLIND:.1f}%")

That `calibrate()` → `predict_concentration()` pair is a small, honest method you
can reuse: it fits the line, reports the statistics that matter, sets detection
limits, and returns predictions *with* uncertainty — graded here against a known
answer it never saw.

## Key Takeaways

- **A calibration curve is an inverse tool.** You measure signal and want
  concentration; the line is what lets you run the instrument backwards. Its
  slope is sensitivity, its intercept is the zero-concentration signal.
- **Beer–Lambert is why the line exists** — and its limits are why the line
  eventually bends. Calibrate to measure the relationship locally; never assume
  it holds outside the range you checked.
- **R² is necessary but not sufficient.** A curved response can post R² > 0.99
  and still be systematically wrong. The **residual plot** is the real test: no
  structure means a trustworthy model; an arch, fan, or drift means it is not.
- **Always report uncertainty on a predicted concentration.** The inverse-
  prediction interval shrinks with more replicates of the unknown and more
  standards, and widens toward the edges of the range. A value without an
  interval is half a result.
- **The blank sets the floor.** LOD ≈ 3.3 σ/slope and LOQ ≈ 10 σ/slope. Report
  readings below LOQ as `< LOQ`, not as defended numbers. Higher sensitivity
  (bigger slope) lowers both limits.
- **Replicates and blanks are not optional.** They are what turn a line into a
  method with a known noise floor and an honest error bar.

## Practical Checklist

- [ ] **Bracket the expected unknowns** with standards — the unknown should fall
  *inside* the calibrated range, ideally near the middle.
- [ ] **Use replicates** at each level, and **measure blanks** for the detection
  limit.
- [ ] **Baseline-correct each measurement** the same way (here, a local linear
  drop) so the intercept reflects chemistry, not background.
- [ ] **Always plot the residuals**, not just R². Look for any concentration-
  dependent pattern.
- [ ] **Check the intercept** is near zero (or explain why it isn't).
- [ ] **Keep quantified results at or above the LOQ**; flag anything below.
- [ ] **Report the prediction with its confidence interval**, and state the
  calibrated range.

## Common Mistakes

- **Reading R² and stopping.** A high R² with curved residuals is a biased
  calibration wearing a good disguise.
- **Extrapolating past the highest standard.** The response may already have
  bent; the line will read confidently and wrongly.
- **Forcing the line through zero** without checking the intercept, hiding a real
  background offset.
- **Calibrating from single points** with no replicates, so there is no estimate
  of scatter and therefore no honest uncertainty or detection limit.
- **Reporting a number below the LOQ** as if it were quantitative.
- **Quoting a predicted concentration with no uncertainty** — it looks precise,
  but you cannot tell whether the last digits mean anything.

## Reporting Guidance

State the method completely enough to reproduce: the **signal measured** (e.g.
"baseline-corrected peak height at 520 nm over a ±1.5·FWHM window"), the
**calibration** (slope ± SE, intercept ± SE, R², number of standards and
replicates, and the **concentration range**), the **detection and quantitation
limits** with how they were determined (blank σ vs residual s_y/x, and the 3.3 /
10 factors), and for each unknown the **predicted concentration with its
confidence interval** and the number of replicate readings. If the response is
nonlinear, say so and report the model and range used instead of pretending a
line fit.

## Next Lesson

**4.3 — NIR Preprocessing: SNV, MSC, and Scatter Correction.** A clean UV-Vis
band gave us one number that scaled neatly with concentration. Near-infrared
spectra rarely cooperate: overlapping overtones, sloping baselines, and
light-scattering effects mean you must *preprocess* the spectrum before any
calibration is trustworthy. That preprocessing — and why it is the gateway to the
multivariate calibrations of Track 6 — is where we go next.